In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

summary = pd.read_csv("../data/multi_pair_summary.csv")
per_pair = pd.read_csv("../data/multi_pair_per_pair_equity.csv", index_col=0, parse_dates=True)
portfolio = pd.read_csv("../data/multi_pair_portfolio_equity.csv", index_col=0, parse_dates=True)["portfolio_equity"]

print("Per-pair summary (full history):")
print(summary.to_string(index=False))
print()
print(f"Portfolio range: {portfolio.index.min().date()} to {portfolio.index.max().date()}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
colors = {"NDSN/OTIS": "navy", "CARR/TT": "crimson", "MA/V": "darkgreen",
          "EOG/FANG": "darkorange", "TRGP/WMB": "purple"}
for col in per_pair.columns:
    per_pair[col].plot(ax=ax, label=col, linewidth=1.5, color=colors.get(col))
ax.axhline(20_000, color="black", linestyle="--", alpha=0.4, label="Initial $20k allocation")
ax.set_title("Per-pair equity (common window, equal $20k allocation each)")
ax.set_ylabel("Equity ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
portfolio.plot(ax=ax, color="navy", linewidth=1.5, label="5-pair equal-weight portfolio")
ax.axhline(100_000, color="black", linestyle="--", alpha=0.4, label="Initial $100k")
ax.set_title(f"Portfolio equity — Sharpe {-0.30:.2f}, Return {-1.10:.2f}%")
ax.set_ylabel("Equity ($)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Reconstruct per-step Sharpes from saved data
per_step_data = {
    "NDSN/OTIS": [0.37, 2.31, -0.71, -0.45, 0.99],
    "CARR/TT":   [-0.16, 0.23, 1.16, -2.67, 0.73],
    "MA/V":      [0.67, -0.72, 1.68, 0.33, -0.98, -0.11, 2.29, -0.13, -0.75, -0.04, 1.03, 2.94, 0.03, 3.55, -0.53],
    "EOG/FANG":  [2.71, 0.68, 2.01, 0.93, 2.12, 0.42, -1.15, 1.53, 1.21, 1.74, 0.66, -0.14, 0.42, -0.72, -2.45],
    "TRGP/WMB":  [-2.15, 1.65, -0.60, -1.41, 3.46, 2.02, -1.26, -1.88, 2.30, 0.16, 0.59, 0.12, 0.31, 2.36, -2.74],
}

# Align by step index from the END (most recent step on the right)
# Each pair's last step is its most recent OOS window
max_steps = max(len(v) for v in per_step_data.values())
aligned = pd.DataFrame(index=range(-max_steps + 1, 1), columns=list(per_step_data.keys()), dtype=float)
for pair, sharpes in per_step_data.items():
    for i, s in enumerate(sharpes[::-1]):
        aligned.loc[-i, pair] = s

import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(13, 4))
cmap = plt.cm.RdYlGn  # red-yellow-green
norm = mcolors.TwoSlopeNorm(vmin=-3, vcenter=0, vmax=3)
data = aligned.T.values  # pairs x steps
im = ax.imshow(data, aspect="auto", cmap=cmap, norm=norm)
ax.set_yticks(range(len(aligned.columns)))
ax.set_yticklabels(aligned.columns)
ax.set_xticks(range(len(aligned.index)))
ax.set_xticklabels([f"t{int(s)}" for s in aligned.index])
ax.set_xlabel("Step index relative to most recent (t0 = latest)")
ax.set_title("Per-step out-of-sample Sharpe by pair (red = loss, green = win)")

# Annotate cells with values
for i, pair in enumerate(aligned.columns):
    for j, step in enumerate(aligned.index):
        val = aligned.loc[step, pair]
        if not pd.isna(val):
            ax.text(j, i, f"{val:.1f}", ha="center", va="center",
                    fontsize=8, color="black")

plt.colorbar(im, label="Sharpe")
plt.tight_layout()
plt.show()